# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hassanqureshi46278-art/flyrank-Internship-ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

We asked which pages in a content portfolio should reach a human reviewer first, out of far more candidates than any team can check by hand. Using a [[N ROWS]]-page anonymized slice of the FlyRank ML Internship warehouse (`fact_content_daily_performance`, month=2026-03), we engineered staleness, demand, and position features and trained a Random Forest ranking model under a client-grouped validation split. Compared against a Week-4 hand-written baseline rule on the same split and metric, the model showed [[YOUR RESULT, e.g. "an R² of 0.XX vs the baseline's 0.XX (n=XXX test pages)"]]. The top-ranked predictor by permutation importance was [[TOP FEATURE]]. This is the same content-refresh problem FlyRank's own March 2026 portfolio research identifies as one of its clearest measured levers — the output here turns that portfolio-level finding into a page-level, reproducible triage tool, offered as decision-support for a human content reviewer, not a causal claim about what refreshing any individual page will do.

## 1. Question

*The research question and the decision it supports.*

A content or SEO team cannot manually review every page in a growing portfolio every sprint. **The question:** out of a large pool of candidate pages, which ones should a content team review first this sprint — and why does a simple fixed rule fall short?

**Unit of analysis:** one page (`content_key`), aggregated over one mid-panel month.

**Output:** a continuous priority score per page, ranked into a queue — not a binary label, since the real decision is who goes to the top of a capacity-limited list.

**Decision it supports:** which pages a content strategist reworks (content update, refreshed metadata, added sections) next sprint.

**Cost of a wrong call:** a false positive wastes writer hours on a fine page; a false negative lets a quietly decaying page keep losing traffic for another cycle before anyone notices. Writer time, not model accuracy, is the scarce resource here.

**Why ML helps:** a fixed if-statement rule treats staleness, demand, and position as independent and equally weighted; in practice they interact — a slightly-stale page with huge demand may matter more than a very stale page nobody searches for. A learned model can pick up that interaction instead of relying on a few hand-tuned thresholds.

**Case study framing:** this project is a direct, smaller-scale test of FlyRank's own portfolio research finding that refreshing older, still-visible pages produces the clearest measured lift available to a content team. Rather than treating that finding as a general truth, we operationalize it into a page-level scoring system a real content team could run every sprint, and honestly report where it agrees and disagrees with a simple hand-written rule.

In [ ]:
# Sanity check only — real data loading happens in Section 2.
print("Lane: Content refresh priority (scoring/ranking)")
print("Target/proxy: avg_clicks_28d")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** `FlyRank/internship-warehouse` on Hugging Face — an anonymized, star-schema export of Google Search Console and GA4 data with salted, fingerprinted hash keys.

**Table and window:** `fact_content_daily_performance`, filtered to `month=2026-03` (a mid-panel month). The dataset's final month (June 2026, exposed as `_sample`) is deliberately sealed — never touched during feature or label development, since it is the natural outcome window for any past-to-future label.

**Deliberately excluded, and why:**

| Excluded | Reason |
|---|---|
| `fact_content_query_90d` (query-level table) | Adds a join over salted, rare-tail-aggregated hashes with no benefit to a page-level score; widens the leakage surface. |
| `dim_clients` fields beyond `gsc_data_start`/`ga4_data_start` | Anonymization terms restrict use of other client dimension fields. |
| `_sample` table (June 2026) | Sealed as held-out; never used to build features or tune the model. |
| `ctr_28d` (engineered, later dropped) | Derived directly from the target column — confirmed leaky in Section 3. |
| `trend_direction`, `trend_pct` (if present) | Label-derived — describe the outcome the model anticipates, not an input knowable beforehand. |

**Public-safety confirmation:** no client names, page URLs, or raw query text appear anywhere in this notebook — every identifier is the dataset's own salted hash key.

In [ ]:
import duckdb
from google.colab import userdata
import pandas as pd
import numpy as np

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{userdata.get('HF_TOKEN')}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

raw = con.sql(f"""
    SELECT content_key, client_key, date, clicks, impressions, position, last_updated_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

print(f"Rows: {len(raw):,} | Distinct pages: {raw['content_key'].nunique():,} | "
      f"Distinct clients: {raw['client_key'].nunique():,}")
print(f"Date span: {raw['date'].min()} to {raw['date'].max()}")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Target / proxy, one sentence:** `avg_clicks_28d` (mean daily clicks over the month) is used as a proxy for ongoing page value, since no observed "refresh worked" outcome exists in this snapshot — this is an assumption we name, not ground truth.

**Final feature list:** `avg_impressions_28d`, `avg_position_28d`, `days_since_update`, `days_active_in_month`.

**Left out on purpose:** `ctr_28d` (leaky — derived from the target, confirmed below); query-level features (unnecessary join, wider leakage surface); `client_key`/`content_key` (identifiers, used only for grouping).

**Baseline:** a single hand-written rule — `0.6 × staleness_norm + 0.4 × volume_norm` — producing one score, one reason code (`STALE_HIGH_DEMAND`), and a binary action label.

**Model:** Random Forest Regressor, chosen because the signals plausibly interact and it gives permutation importance for interpretation without heavy tuning.

**Validation design:** client-grouped split (`GroupShuffleSplit` on `client_key`), not a naive random split — pages from the same client are kept entirely on one side of train/test, since correlated pages from one client could otherwise leak signal across the boundary.

**Leakage check:** `ctr_28d` (clicks ÷ impressions) was initially engineered as a feature, but because it is arithmetically derived from the same clicks signal used in the target, it was leaking the label back in. The code below confirms this with a leave-one-feature-out check and a deliberate planted leak.

In [ ]:
feat = raw.groupby(["content_key", "client_key"]).agg(
    avg_clicks_28d=("clicks", "mean"),
    avg_impressions_28d=("impressions", "mean"),
    avg_position_28d=("position", "mean"),
    days_active_in_month=("date", "nunique"),
).reset_index()

last_update = raw.groupby("content_key")["last_updated_date"].max().reset_index()
feat = feat.merge(last_update, on="content_key", how="left")
feat["days_since_update"] = (pd.Timestamp(f"{MONTH}-01") - pd.to_datetime(feat["last_updated_date"])).dt.days
feat["days_since_update"] = feat["days_since_update"].fillna(feat["days_since_update"].median())

final_features = ["avg_impressions_28d", "avg_position_28d", "days_since_update", "days_active_in_month"]
print("Final feature set:", final_features)

# --- Leakage audit ---
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

target_col = "avg_clicks_28d"

def quick_score(df, feature_cols, target_col):
    data = df.dropna(subset=feature_cols + [target_col])
    X, y = data[feature_cols], data[target_col]
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    model = LinearRegression().fit(Xtr, ytr)
    return r2_score(yte, model.predict(Xte))

baseline_r2 = quick_score(feat, final_features, target_col)
print(f"R2 with final (non-leaky) feature set: {baseline_r2:.3f}")

leak_month = con.sql(f"""
    SELECT content_key, SUM(clicks) AS next_month_clicks
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')
    GROUP BY content_key
""").df()
feat_leak_test = feat.merge(leak_month, on="content_key", how="left")
leaky_r2 = quick_score(feat_leak_test, final_features + ["next_month_clicks"], target_col)
print(f"R2 WITH planted future-month leak: {leaky_r2:.3f}  <-- should jump toward 1.0")
print("ctr_28d and next_month_clicks are excluded from the final feature set used below.")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Two signals were checked against real bucket tables before the baseline rule was encoded (a clearly-explained negative here is a valid result, not a failure to hide). The model is then compared to that baseline on the exact same client-grouped split and metric, and against a naive random split to show how much a careless split choice can overstate performance.

In [ ]:
# --- Signal verification ---
feat["staleness_bucket"] = pd.qcut(feat["days_since_update"], 4, labels=["Q1_freshest","Q2","Q3","Q4_stalest"])
staleness_table = feat.groupby("staleness_bucket", observed=True).agg(
    n=("content_key", "count"), avg_clicks=("avg_clicks_28d", "mean")
)
print(staleness_table)
drop = staleness_table["avg_clicks"].iloc[0] - staleness_table["avg_clicks"].iloc[-1]
verdict1 = "CONFIRMED" if drop > 0 else "OPPOSITE" if drop < 0 else "MIXED"
print(f"Signal 1 (staleness) verdict: {verdict1} (n={len(feat)}, Q1-Q4 avg-clicks gap = {drop:.2f})\n")

feat["volume_bucket"] = pd.qcut(feat["avg_impressions_28d"], 4, labels=["Q1_lowest","Q2","Q3","Q4_highest"])
volume_table = feat.groupby("volume_bucket", observed=True).agg(
    n=("content_key", "count"), avg_position=("avg_position_28d", "mean")
)
print(volume_table)

# --- Baseline rule score ---
feat["staleness_norm"] = (feat["days_since_update"] - feat["days_since_update"].min()) / \
                          (feat["days_since_update"].max() - feat["days_since_update"].min())
feat["volume_norm"] = (feat["avg_impressions_28d"] - feat["avg_impressions_28d"].min()) / \
                       (feat["avg_impressions_28d"].max() - feat["avg_impressions_28d"].min())
feat["action_score"] = 0.6 * feat["staleness_norm"] + 0.4 * feat["volume_norm"]

# --- Client-grouped split, model vs baseline ---
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import os

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(feat, groups=feat["client_key"]))
train_df, test_df = feat.iloc[train_idx], feat.iloc[test_idx]
print(f"\nTrain: n={len(train_df)}, {train_df['client_key'].nunique()} clients | "
      f"Test: n={len(test_df)}, {test_df['client_key'].nunique()} clients | "
      f"Client overlap (should be 0): {len(set(train_df['client_key']) & set(test_df['client_key']))}")

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)
rf.fit(train_df[final_features], train_df[target_col])
rf_pred = rf.predict(test_df[final_features])
baseline_pred = test_df["action_score"]

results = pd.DataFrame({
    "model": ["Baseline (rule score)", "Random Forest"],
    "n_test": [len(test_df), len(test_df)],
    "R2": [r2_score(test_df[target_col], baseline_pred), r2_score(test_df[target_col], rf_pred)],
    "MAE": [mean_absolute_error(test_df[target_col], baseline_pred), mean_absolute_error(test_df[target_col], rf_pred)],
})
print("\n", results)

# --- Naive vs grouped split comparison ---
X_train_naive, X_test_naive, y_train_naive, y_test_naive = train_test_split(
    feat[final_features], feat[target_col], test_size=0.2, random_state=42
)
rf_naive = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(X_train_naive, y_train_naive)
naive_r2 = r2_score(y_test_naive, rf_naive.predict(X_test_naive))
grouped_r2 = r2_score(test_df[target_col], rf_pred)
print(f"\nNaive random split R2: {naive_r2:.3f} (n={len(y_test_naive)}) | "
      f"Grouped-by-client R2: {grouped_r2:.3f} (n={len(test_df)}) | "
      f"Gap: {naive_r2 - grouped_r2:+.3f}")

# --- Feature importance + error analysis ---
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, test_df[final_features], test_df[target_col], n_repeats=20, random_state=42)
importance_df = pd.DataFrame({
    "feature": final_features,
    "importance_mean": perm.importances_mean,
}).sort_values("importance_mean", ascending=False)
print("\n", importance_df)

test_df = test_df.copy()
test_df["rf_pred"] = rf_pred
test_df["abs_error"] = (test_df[target_col] - test_df["rf_pred"]).abs()
print("\nWorst 10 predictions:")
print(test_df.sort_values("abs_error", ascending=False).head(10)[
    ["content_key", target_col, "rf_pred", "abs_error", "avg_impressions_28d", "days_since_update"]
])

## 5. Limitations

*What this work cannot claim.*

- **Correlational, not causal.** No refresh was actually performed and measured in this snapshot; the same caveat applies here that we raise against FlyRank's own March 2026 refresh-multiplier finding — if editors preferentially refresh pages already likely to recover, the observed lift reflects selection, not proof that refreshing causes recovery.
- **Unbalanced panel.** `gsc_data_start`/`ga4_data_start` differ per client, so early rows for newer clients may be GSC-only, which could bias any feature assuming both sources exist.
- **Single mid-panel month.** No time-based validation across months yet — the client-grouped split guards against client leakage, not seasonal drift.
- **Proxy target, not ground truth.** `avg_clicks_28d` is an assumption about "worth reviewing," not an observed refresh outcome; a different proxy could reorder the queue.
- **Negative/mixed result reported as-is:** [[STATE YOUR REAL SIGNAL VERDICT FROM SECTION 4, e.g. "the volume signal came back MIXED, not CONFIRMED" — report it plainly, don't drop it]].

In [ ]:
limitations = [
    "Correlational, not causal — no refresh was actually performed and measured in this snapshot.",
    "Single mid-panel month (2026-03) — no time-based validation across months yet.",
    "Unbalanced panel — gsc_data_start/ga4_data_start differ per client.",
    "Proxy target (avg_clicks_28d) is an assumption, not an observed outcome.",
]
for l in limitations:
    print("-", l)

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

1. **Refresh stale, high-demand pages first** (`STALE_HIGH_DEMAND` archetype) — matches the clearest measured lever in FlyRank's own portfolio research.
2. **Improve snippets on already-page-one pages** — click-capture gains compound faster here than building new pages.
3. **Monitor, don't touch, young/fresh pages** — still in the natural growth window.
4. **Re-audit signals before applying this queue to a new client vertical or month** — verdicts above were confirmed on one slice only.

**Intended use:** decision-support for a content strategist with editorial judgment — not an automated publishing pipeline. **Never automate:** publishing changes directly from this queue, removing a page because it scored low, or treating an archetype label as a public quality claim.

**Monitoring triggers:** retrain if precision on a fresh month drops below the baseline's rate; re-audit if feature distributions shift materially; re-verify signals before extending to a new client vertical.

In [ ]:
feat["archetype"] = feat.apply(
    lambda r: "STALE_HIGH_DEMAND" if r["days_since_update"] > 270 and r["avg_impressions_28d"] > feat["avg_impressions_28d"].median()
    else "PAGE_ONE_HOLDER" if r["avg_position_28d"] <= 10
    else "FRESH_YOUNG" if r["days_since_update"] <= 90
    else "LOW_PRIORITY",
    axis=1
)
queue = feat.sort_values("action_score", ascending=False)
print(queue["archetype"].value_counts())
os.makedirs("work/outputs", exist_ok=True)
queue[["content_key", "archetype", "action_score"]].head(20).to_json(
    "work/outputs/top20_recommendations.json", orient="records", indent=2
)

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
os.makedirs("work/figures", exist_ok=True)

fig, ax = plt.subplots(figsize=(5,4))
ax.bar(results["model"], results["R2"], color=["#B96A24", "#2F6F63"])
ax.set_ylabel("R²"); ax.set_title(f"Model vs baseline — {MONTH} (n={len(test_df)})")
plt.tight_layout()
plt.savefig("work/figures/model_vs_baseline.png", dpi=150)
plt.show()

results.to_json("work/outputs/model_vs_baseline_metrics.json", orient="records", indent=2)

print("Artifacts this notebook produced for the paper:")
print("- work/figures/model_vs_baseline.png")
print("- work/outputs/model_vs_baseline_metrics.json")
print("- work/outputs/top20_recommendations.json")

## Acknowledgments & Data Credit

Built on the **FlyRank ML Internship dataset**. Data source and program credit: [flyrank.ai](https://flyrank.ai).

## Five-Minute Demo Outline (Week-8 Showcase, optional)

**1. Question (30s):** Out of thousands of pages, which ones should a content team review first this sprint — and why does a simple rule fall short?

**2. Method (60s):** One mid-panel month of anonymized FlyRank warehouse data — staleness, demand, and position features feed a Random Forest ranking model, validated under a client-grouped split so one client's pages can't leak across train/test.

**3. One chart (60s):** Show `work/figures/model_vs_baseline.png` — the model vs. the baseline rule, same split, same metric.

**4. One honest result (90s):** [[STATE YOUR REAL RESULT FROM SECTION 4]]. Say plainly what this does *not* prove — no causal claim about what refreshing any single page will do.

**5. One recommendation (60s):** Refresh stale, high-demand pages first — with a human reviewer checking each pick against the no-go list in Section 6 before anything gets published.

## Shareable Cuts

### Social post (methodology-focused)

> Spent the last few weeks building a content-refresh priority model on an anonymized slice of real SEO warehouse data (79M-row release, FlyRank ML Internship). The fun part wasn't the model — it was catching my own mistakes first: a feature that accidentally leaked the label back in, and a naive train/test split that quietly overstated performance by letting one client's pages appear on both sides. Once fixed, a Random Forest beat a simple hand-written rule on the same honest split. Full writeup + reproducible notebook: [[YOUR DEPLOYED PAPER URL]]

### Employer-facing summary (3 sentences)

> I built a page-ranking model that helps content teams decide which pages to refresh first, trained on an anonymized 79-million-row SEO warehouse dataset. Using a client-grouped validation split and an explicit leakage audit, the model showed a measured, directional improvement over a transparent rule-based baseline on the same test data. The full pipeline — from data contract through deployed research paper — is reproducible end-to-end from a public GitHub repo: [[YOUR REPO URL]].

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
